# 02 — Data cleaning & genre one-hot encoding

Stage 2 of the pipeline. Takes the raw table produced by `movie_pipeline.extract` and produces the
analysis-ready dataset used by the recommender.

| Step | What | Why |
|---|---|---|
| 1 | Drop columns > 50 % missing | too sparse to be useful |
| 2 | Strip whitespace, coerce `rating` / `viewers` / `total_sales` to numbers | the archive uses Persian thousands separators (`٬`) |
| 3 | Drop duplicate rows | the same card can appear twice in a saved page |
| 4 | Multi-label one-hot encode `genres` → `genre_*` | a movie can have several genres |
| 5 | Remove `rating == 10` rows | data-entry artefact on a 5-point scale |

> The same logic is packaged as `movie_pipeline.clean.clean_movies()` (unit-tested in `tests/test_clean.py`).
> This notebook keeps the exploratory version so each intermediate result can be inspected.

---

*پاک‌سازی داده‌ها و وان‌هات‌انکود کردن ژانرها*

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "all_years_movies.csv"
OUT = ROOT / "data" / "processed" / "all_years_movies_cleaned_ohe.csv"

df = pd.read_csv(RAW)
print(df.shape)
df.head()

## 1) Drop sparse columns

In [ ]:
missing_pct = df.isna().mean() * 100
cols_to_drop = missing_pct[missing_pct > 50].index.tolist()
print("dropping:", cols_to_drop)
df_clean = df.drop(columns=cols_to_drop)
missing_pct.round(1)

## 2) Normalise text and numeric columns

In [ ]:
for col in df_clean.columns:
    if df_clean[col].dtype == object:
        df_clean[col] = df_clean[col].str.strip()


def to_numeric(series: pd.Series) -> pd.Series:
    """Remove Persian/Latin thousands separators and coerce to float (bad values → NaN)."""
    return pd.to_numeric(series.astype(str).str.replace(r"[٬, ]", "", regex=True), errors="coerce")


for num in ["rating", "viewers", "total_sales"]:
    if num in df_clean.columns and df_clean[num].dtype == object:
        df_clean[num] = to_numeric(df_clean[num])

df_clean = df_clean.drop_duplicates()
df_clean.dtypes

## 3) Multi-label one-hot encoding of `genres`

`genres` is a comma-separated string such as `اجتماعی, درام, کمدی`. Each distinct genre becomes a binary `genre_<name>` column.

In [ ]:
def split_genres(value) -> set[str]:
    if pd.isna(value):
        return set()
    return {g.strip() for g in str(value).split(",") if g.strip()}


genre_sets = df_clean["genres"].map(split_genres)
unique_genres = sorted({g for s in genre_sets for g in s})
print(len(unique_genres), "genres:", unique_genres)

for g in unique_genres:
    df_clean[f"genre_{g}"] = genre_sets.map(lambda s, g=g: int(g in s))

df_clean.filter(like="genre_").sum().sort_values(ascending=False).head(10)

## 4) Outlier inspection

Ratings are on a 1–5 scale, so any value of 10 is an artefact of the source page and is removed.

In [ ]:
df_clean.describe()

In [ ]:
before = len(df_clean)
df_clean = df_clean[df_clean["rating"] != 10]
print(f"removed {before - len(df_clean)} outlier rows → {len(df_clean)} movies")

## 5) Save the processed dataset

In [ ]:
OUT.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(OUT, index=False, encoding="utf-8-sig")
print(f"✅ saved {df_clean.shape[0]} rows × {df_clean.shape[1]} columns → {OUT.relative_to(ROOT)}")